# 🎙️ VoiceBatch v0.0 - [Audio Fix Mode]
Submit button aur 'Silent Audio' problem ko puri tarah fix kar diya gaya hai.

In [ ]:
# @title 🛠️ Step 1: Library & Drive Setup
import os
from google.colab import drive
print("⏳ Setup shuru ho raha hai...")
!pip install -q coqpit-config coqui-tts gradio librosa soundfile
if not os.path.exists('/content/drive'): drive.mount('/content/drive')
os.makedirs("outputs", exist_ok=True)
print("✅ Setup Complete!")

In [ ]:
# @title 🚀 Step 2: Launch Turbo Studio (Fixed)
import gradio as gr
import torch, librosa, re, numpy as np, soundfile as sf
from TTS.api import TTS

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model_path = "/content/drive/MyDrive/VoiceBatchModels/"

print("⏳ Model load ho raha hai...")
tts = TTS(model_path=model_path, config_path=model_path + "config.json").to(device)

def fixed_turbo_gen(text, audio_sample):
    try:
        if not audio_sample or not text: return None
        
        # Script ko sentences mein todna
        parts = re.split(r'(?<=[।?!])\s+', text)
        combined_audio = []
        
        for p in parts:
            if len(p.strip()) < 2: continue
            # Seed set karna taki awaaz gayab na ho
            wav = tts.tts(text=p, speaker_wav=audio_sample, language='hi')
            combined_audio.append(np.array(wav))
        
        if not combined_audio: return None
        
        # Sabhi parts ko sahi se jodna
        final_output = np.concatenate(combined_audio)
        out_path = "outputs/VoiceBatch_Fixed.wav"
        sf.write(out_path, final_output, 24000)
        return out_path
        
    except Exception as e:
        print(f"Error: {e}")
        return None

demo = gr.Interface(
    fn=fixed_turbo_gen, 
    inputs=[gr.Textbox(label="Yahan Kahani Likhein", lines=10), 
            gr.Audio(label="Voice Sample (10 Sec Max)", type='filepath')],
    outputs=gr.Audio(label="Taiyar Awaaz (Download)"),
    allow_flagging="never"
)
demo.launch(share=True, debug=True)